# Nairobi DL-Only Diagnostics

Apply the best trained Sen1Floods11 deep-learning water model to Nairobi before running any fusion model.

Run this notebook step-by-step. It produces before/after water probability, a DL flood proxy (`after_water & ~before_water`), an overlap summary with the available ground-truth polygons, and a quicklook figure.


In [ ]:
# Imports
import os
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import odc.stac
import pandas as pd
import pystac_client
import rasterio
import rioxarray  # noqa: F401; registers .rio accessor
import torch
import torch.nn as nn
import xarray as xr
from odc.geo.geobox import GeoBox
from rasterio.features import rasterize
from rasterio.transform import from_origin
from shapely.geometry import box


In [ ]:
# Configuration
# Works whether the notebook kernel starts in the repo root or inside FinalProjects.
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent
elif not (PROJECT_DIR / "data").exists() and (PROJECT_DIR / "FinalProjects" / "data").exists():
    PROJECT_DIR = PROJECT_DIR / "FinalProjects"
print("PROJECT_DIR:", PROJECT_DIR)

RUN_NAME = "exp06_resattn_auto_wce_dice025_lr8e-4_base32_e80_pat20"
MODEL_CHECKPOINT = PROJECT_DIR / "results" / "outputs_DL" / RUN_NAME / "sen1floods11_small_unet_best.pt"
BEST_WATER_THRESHOLD = 0.45

OUT_DIR = PROJECT_DIR / "results" / "outputs" / "nairobi_deep_learning_dl_only"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DX = 0.0003  # about 30 m in EPSG:4326 near Nairobi
EPSG = 4326
LATMIN, LATMAX = -1.5371, -1.0918
LONMIN, LONMAX = 36.5134, 37.2744
BOUNDS = (LONMIN, LATMIN, LONMAX, LATMAX)
DATE_QUERY = "2024-03-06/2024-06-01"
BEFORE_SLICE = slice("2024-03-06", "2024-04-06")
AFTER_SLICE = slice("2024-05-01", "2024-06-01")
TILE_SIZE = 512
TILE_STRIDE = 384

print("Checkpoint:", MODEL_CHECKPOINT)
print("Output folder:", OUT_DIR)


## Model Selection Summary

The Nairobi DL-only workflow uses the best Sen1Floods11 validation run as the transfer model. The selected checkpoint is **Exp 6: Residual Attention U-Net with auto weighted cross entropy and Dice loss (0.25)**. This model was selected because it achieved the highest global validation water IoU and F1 score, avoided the all-background collapse mode, and produced a predicted water fraction close to the true validation water fraction.

The recommended validation decision rule is the global softmax water-threshold sweep, not the batch-averaged argmax metric, because it computes one confusion matrix over all valid validation pixels and directly supports threshold selection for transfer inference.


In [ ]:
# Load the report-ready experiment table used for model selection.
experiment_table_path = PROJECT_DIR / "results" / "experiments" / "dl_experiment_report_table.csv"
experiment_table = pd.read_csv(experiment_table_path)

report_columns = [
    "rank_by_global_iou",
    "report_label",
    "model",
    "loss",
    "learning_rate",
    "checkpoint_epoch",
    "best_threshold",
    "global_iou_water",
    "global_f1_water",
    "precision",
    "recall",
    "pred_water_frac",
    "collapse_check",
]
display(experiment_table[report_columns].round(3))

selected_model = experiment_table.iloc[0]
print("Selected run:", selected_model["run_name"])
print("Selected threshold:", selected_model["best_threshold"])


In [ ]:
# Sentinel-1 preprocessing helpers

def spatial_dims(da):
    non_time_dims = [d for d in da.dims if d != "time"]
    y_dim = "y" if "y" in da.dims else non_time_dims[-2]
    x_dim = "x" if "x" in da.dims else non_time_dims[-1]
    return y_dim, x_dim


def to_db(da):
    return 10 * np.log10(da.where(da > 0))


def remove_column_brightness_bias(da_db, window=121):
    y_dim, x_dim = spatial_dims(da_db)
    scene_median = da_db.median(dim=(y_dim, x_dim), skipna=True)
    column_profile = da_db.median(dim=y_dim, skipna=True)
    smooth_profile = column_profile.rolling({x_dim: window}, center=True, min_periods=1).median()
    correction = smooth_profile - scene_median
    return da_db - correction


def normalize_s1_tile(x):
    x = x.astype("float32")
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    out = np.empty_like(x, dtype="float32")
    for band_idx in range(x.shape[0]):
        band = x[band_idx]
        lo, hi = np.percentile(band, [2, 98])
        if hi <= lo:
            out[band_idx] = 0.0
        else:
            out[band_idx] = np.clip((band - lo) / (hi - lo), 0.0, 1.0)
            out[band_idx] = out[band_idx] * 2.0 - 1.0
    return out


In [ ]:
# Model definitions and checkpoint loading
# Supports both the original SmallUNet checkpoints and the stronger residual-attention U-Net checkpoints.

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class SmallUNet(nn.Module):
    def __init__(self, in_channels=2, num_classes=2, base=32):
        super().__init__()
        self.enc1 = DoubleConv(in_channels, base)
        self.enc2 = DoubleConv(base, base * 2)
        self.enc3 = DoubleConv(base * 2, base * 4)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(base * 4, base * 8)
        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, stride=2)
        self.dec3 = DoubleConv(base * 8, base * 4)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = DoubleConv(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = DoubleConv(base * 2, base)
        self.head = nn.Conv2d(base, num_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b = self.bottleneck(self.pool(e3))
        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.head(d1)


class ResidualConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout2d(dropout) if dropout > 0 else nn.Identity()
        self.shortcut = nn.Identity() if in_channels == out_channels else nn.Conv2d(in_channels, out_channels, 1, bias=False)

    def forward(self, x):
        identity = self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.dropout(out)
        out = self.bn2(self.conv2(out))
        return self.relu(out + identity)


class AttentionGate(nn.Module):
    def __init__(self, gating_channels, skip_channels, inter_channels):
        super().__init__()
        self.gate = nn.Sequential(nn.Conv2d(gating_channels, inter_channels, 1, bias=False), nn.BatchNorm2d(inter_channels))
        self.skip = nn.Sequential(nn.Conv2d(skip_channels, inter_channels, 1, bias=False), nn.BatchNorm2d(inter_channels))
        self.psi = nn.Sequential(nn.Conv2d(inter_channels, 1, 1), nn.Sigmoid())
        self.relu = nn.ReLU(inplace=True)

    def forward(self, gating, skip):
        attention = self.psi(self.relu(self.gate(gating) + self.skip(skip)))
        return skip * attention


class AttentionUpBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, dropout=0.0):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, out_channels, 2, stride=2)
        self.attn = AttentionGate(out_channels, skip_channels, max(out_channels // 2, 1))
        self.conv = ResidualConvBlock(out_channels + skip_channels, out_channels, dropout=dropout)

    def forward(self, x, skip):
        x = self.up(x)
        skip = self.attn(x, skip)
        return self.conv(torch.cat([x, skip], dim=1))


class ResidualAttentionUNet(nn.Module):
    def __init__(self, in_channels=2, num_classes=2, base=32, dropout=0.10):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.enc1 = ResidualConvBlock(in_channels, base, dropout=0.0)
        self.enc2 = ResidualConvBlock(base, base * 2, dropout=dropout)
        self.enc3 = ResidualConvBlock(base * 2, base * 4, dropout=dropout)
        self.enc4 = ResidualConvBlock(base * 4, base * 8, dropout=dropout)
        self.bottleneck = ResidualConvBlock(base * 8, base * 16, dropout=dropout)
        self.up4 = AttentionUpBlock(base * 16, base * 8, base * 8, dropout=dropout)
        self.up3 = AttentionUpBlock(base * 8, base * 4, base * 4, dropout=dropout)
        self.up2 = AttentionUpBlock(base * 4, base * 2, base * 2, dropout=dropout)
        self.up1 = AttentionUpBlock(base * 2, base, base, dropout=0.0)
        self.head = nn.Conv2d(base, num_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))
        d4 = self.up4(b, e4)
        d3 = self.up3(d4, e3)
        d2 = self.up2(d3, e2)
        d1 = self.up1(d2, e1)
        return self.head(d1)


def infer_model_info(checkpoint):
    state = checkpoint["model_state_dict"]
    variant = checkpoint.get("model_variant")
    base = checkpoint.get("base_channels")
    dropout = checkpoint.get("dropout", 0.10)

    # Older residual-attention checkpoints did not save metadata but use enc1.conv1.weight.
    if variant is None:
        if "enc1.conv1.weight" in state:
            variant = "res_attention_unet"
            base = int(state["enc1.conv1.weight"].shape[0])
        elif "enc1.net.0.weight" in state:
            variant = "small_unet"
            base = int(state["enc1.net.0.weight"].shape[0])
        else:
            raise ValueError("Could not infer model variant from checkpoint keys.")

    if base is None:
        if variant == "res_attention_unet":
            base = int(state["enc1.conv1.weight"].shape[0])
        else:
            base = int(state["enc1.net.0.weight"].shape[0])

    return variant, int(base), float(dropout)


def build_model_for_checkpoint(variant, base, dropout):
    if variant == "small_unet":
        return SmallUNet(in_channels=2, num_classes=2, base=base)
    if variant == "res_attention_unet":
        return ResidualAttentionUNet(in_channels=2, num_classes=2, base=base, dropout=dropout)
    raise ValueError(f"Unknown model variant: {variant}")


def load_model(device):
    checkpoint = torch.load(MODEL_CHECKPOINT, map_location=device)
    variant, base, dropout = infer_model_info(checkpoint)
    model = build_model_for_checkpoint(variant, base, dropout).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    print(f"Loaded checkpoint: {MODEL_CHECKPOINT}")
    print(f"Checkpoint epoch: {checkpoint.get('epoch')}")
    print(f"Model variant: {variant}, base_channels={base}, dropout={dropout}")
    print("Checkpoint metrics:", checkpoint.get("metrics"))
    return model, checkpoint


device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", device)
model, checkpoint = load_model(device)


In [ ]:
# Load and preprocess Nairobi Sentinel-1
# This mirrors the working import path from Nairobi Flood FootPrint_angle_corrected.ipynb.
# It keeps the xarray object lazy here; the actual remote reads happen later when median composites are computed.

os.environ["AWS_NO_SIGN_REQUEST"] = "YES"

items = (
    pystac_client.Client.open("https://earth-search.aws.element84.com/v1")
    .search(
        bbox=BOUNDS,
        collections=["sentinel-1-grd"],
        datetime=DATE_QUERY,
        limit=100,
        query={"sar:instrument_mode": {"in": ["IW"]}},
    )
    .item_collection()
)
print(f"Sentinel-1 scenes found: {len(items)}")

geobox = GeoBox.from_bbox(BOUNDS, crs=f"epsg:{EPSG}", resolution=DX)

dc_raw = odc.stac.load(
    items,
    bands=["vv", "vh"],
    geobox=geobox,
    resampling="bilinear",
)

dc_db = xr.Dataset(
    {
        "vv": to_db(dc_raw.vv),
        "vh": to_db(dc_raw.vh),
    },
    attrs=dc_raw.attrs,
)

dc = xr.Dataset(
    {
        "vv": remove_column_brightness_bias(dc_db.vv),
        "vh": remove_column_brightness_bias(dc_db.vh),
    },
    attrs=dc_raw.attrs,
)

before = dc.sel(time=BEFORE_SLICE)
after = dc.sel(time=AFTER_SLICE)

print(f"Before scenes: {before.sizes.get('time', 0)}")
print(f"After scenes: {after.sizes.get('time', 0)}")
dc


In [ ]:
# Build before/after VV/VH median composites in the model's expected band order: [VV, VH]
# This is the point where xarray actually reads the remote Sentinel-1 imagery.

def median_vv_vh(ds):
    vv = ds.vv.median(dim="time", skipna=True).compute()
    vh = ds.vh.median(dim="time", skipna=True).compute()
    return xr.concat([vv, vh], dim=pd.Index(["vv", "vh"], name="band"))

before_img = median_vv_vh(before)
after_img = median_vv_vh(after)
print("Before image shape:", before_img.shape)
print("After image shape:", after_img.shape)
print("Before VV percentiles:", np.nanpercentile(before_img.sel(band="vv").values, [2, 50, 98]).round(3).tolist())
print("After VV percentiles:", np.nanpercentile(after_img.sel(band="vv").values, [2, 50, 98]).round(3).tolist())
print("Before VH percentiles:", np.nanpercentile(before_img.sel(band="vh").values, [2, 50, 98]).round(3).tolist())
print("After VH percentiles:", np.nanpercentile(after_img.sel(band="vh").values, [2, 50, 98]).round(3).tolist())


In [ ]:
# Tiled model inference

def predict_water_probability(image_2band, model, device, tile_size=512, stride=384):
    y_dim, x_dim = spatial_dims(image_2band.isel(band=0))
    image = image_2band.transpose("band", y_dim, x_dim)
    arr = image.values.astype("float32")
    _, h, w = arr.shape

    prob_sum = np.zeros((h, w), dtype="float32")
    weight_sum = np.zeros((h, w), dtype="float32")

    y_starts = list(range(0, max(h - tile_size, 0) + 1, stride))
    x_starts = list(range(0, max(w - tile_size, 0) + 1, stride))
    if not y_starts or y_starts[-1] != max(h - tile_size, 0):
        y_starts.append(max(h - tile_size, 0))
    if not x_starts or x_starts[-1] != max(w - tile_size, 0):
        x_starts.append(max(w - tile_size, 0))

    model.eval()
    with torch.no_grad():
        for y0 in y_starts:
            for x0 in x_starts:
                tile = arr[:, y0:y0 + tile_size, x0:x0 + tile_size]
                valid_h, valid_w = tile.shape[1], tile.shape[2]
                if valid_h < tile_size or valid_w < tile_size:
                    padded = np.zeros((2, tile_size, tile_size), dtype="float32")
                    padded[:, :valid_h, :valid_w] = tile
                    tile = padded
                x = torch.from_numpy(normalize_s1_tile(tile)).unsqueeze(0).to(device)
                logits = model(x)
                prob = torch.softmax(logits, dim=1)[0, 1].cpu().numpy()[:valid_h, :valid_w]
                prob_sum[y0:y0 + valid_h, x0:x0 + valid_w] += prob
                weight_sum[y0:y0 + valid_h, x0:x0 + valid_w] += 1.0

    prob = prob_sum / np.maximum(weight_sum, 1e-6)
    return xr.DataArray(prob, coords=image.isel(band=0).coords, dims=(y_dim, x_dim), name="water_probability")

print("Predicting before water probability...")
before_water_prob = predict_water_probability(before_img, model, device, TILE_SIZE, TILE_STRIDE)
print("Predicting after water probability...")
after_water_prob = predict_water_probability(after_img, model, device, TILE_SIZE, TILE_STRIDE)

before_water_mask = before_water_prob >= BEST_WATER_THRESHOLD
after_water_mask = after_water_prob >= BEST_WATER_THRESHOLD
flood_dl_mask = after_water_mask & (~before_water_mask)
prob_change = after_water_prob - before_water_prob

print("Before water pixels:", int(before_water_mask.sum()))
print("After water pixels:", int(after_water_mask.sum()))
print("DL flood proxy pixels:", int(flood_dl_mask.sum()))


In [ ]:
# Rasterize ground-truth polygons and compute DL-only proxy metrics
# Evaluates DL-only outputs against both:
#   1. Google Research / media-report groundsource polygons
#   2. UNStats flood extent shapefile for Nairobi/Kiambu

UNSTATS_GT_PATH = PROJECT_DIR / "unStat_groundTruth/PL_20240501_FloodExtent_Nairobi_Kiambu.shp"


def rasterize_groundtruth_like(ref, gdf):
    gdf_local = gdf.copy()
    if gdf_local.crs is None:
        gdf_local = gdf_local.set_crs(epsg=EPSG)
    else:
        gdf_local = gdf_local.to_crs(epsg=EPSG)

    shapes = [(geom, 1) for geom in gdf_local.geometry if geom is not None and not geom.is_empty]
    if len(shapes) == 0:
        raise ValueError("No valid geometries to rasterize.")

    x = ref[ref.dims[-1]].values
    y = ref[ref.dims[-2]].values
    xres = float(abs(np.nanmedian(np.diff(x))))
    yres = float(abs(np.nanmedian(np.diff(y))))
    transform = from_origin(float(np.nanmin(x)) - xres / 2, float(np.nanmax(y)) + yres / 2, xres, yres)
    label = rasterize(shapes, out_shape=ref.shape, transform=transform, fill=0, dtype="uint8")
    if y[0] < y[-1]:
        label = np.flipud(label)
    return xr.DataArray(label, coords=ref.coords, dims=ref.dims, name="groundtruth_flood")


# Groundsource / media-report polygons.
groundsource_gt = gpd.read_parquet(PROJECT_DIR / "data/groundsource__2026.parquet")
groundsource_gt["start_date"] = pd.to_datetime(groundsource_gt["start_date"])
groundsource_gt["end_date"] = pd.to_datetime(groundsource_gt["end_date"])
nairobi_box = box(LONMIN, LATMIN, LONMAX, LATMAX)
groundsource_gt = groundsource_gt[
    (groundsource_gt["area_km2"] > 5)
    & (groundsource_gt["area_km2"] <= 600)
    & (groundsource_gt.geometry.within(nairobi_box))
    & (groundsource_gt["start_date"].dt.year == 2024)
].copy()
print("Groundsource polygons:", groundsource_gt.shape[0])

# UNStats flood extent polygon.
unstats_gt = gpd.read_file(UNSTATS_GT_PATH)
if unstats_gt.crs is None:
    unstats_gt = unstats_gt.set_crs(epsg=EPSG)
else:
    unstats_gt = unstats_gt.to_crs(epsg=EPSG)
unstats_gt = unstats_gt[unstats_gt.intersects(nairobi_box)].copy()
print("UNStats polygons:", unstats_gt.shape[0])
print("UNStats bounds:", unstats_gt.total_bounds.round(5).tolist())

# Rasterize each reference and also their union.
gt_groundsource = rasterize_groundtruth_like(after_water_prob, groundsource_gt)
gt_unstats = rasterize_groundtruth_like(after_water_prob, unstats_gt)
gt_combined = ((gt_groundsource.astype(bool)) | (gt_unstats.astype(bool))).astype("uint8")
gt_combined.name = "groundtruth_combined_union"

ground_truth_layers = {
    "groundsource_media_polygons": gt_groundsource,
    "unstats_flood_extent": gt_unstats,
    "combined_union": gt_combined,
}

lat_center = (LATMIN + LATMAX) / 2
pixel_width_m = abs(DX) * 111_320 * np.cos(np.deg2rad(lat_center))
pixel_height_m = abs(DX) * 110_540
pixel_area_km2 = (pixel_width_m * pixel_height_m) / 1_000_000


def summarize_mask(mask_name, mask, gt_name, gt, pixel_area_km2):
    mask = np.asarray(mask).astype(bool)
    gt = np.asarray(gt).astype(bool)
    tp = int((mask & gt).sum())
    fp = int((mask & (~gt)).sum())
    fn = int(((~mask) & gt).sum())
    pixels = int(mask.sum())
    return {
        "ground_truth": gt_name,
        "mask": mask_name,
        "pixel_count": pixels,
        "area_km2_approx": pixels * pixel_area_km2,
        "gt_intersection_pixels": tp,
        "gt_recall_proxy": tp / (tp + fn + 1e-8),
        "gt_precision_proxy": tp / (tp + fp + 1e-8),
        "gt_iou_proxy": tp / (tp + fp + fn + 1e-8),
    }

prediction_masks = {
    "dl_before_water_mask": before_water_mask,
    "dl_after_water_mask": after_water_mask,
    "dl_flood_mask_after_minus_before": flood_dl_mask,
}

rows = []
for gt_name, gt in ground_truth_layers.items():
    rows.append(summarize_mask("ground_truth_reference", gt.values, gt_name, gt.values, pixel_area_km2))
    for mask_name, mask in prediction_masks.items():
        rows.append(summarize_mask(mask_name, mask.values, gt_name, gt.values, pixel_area_km2))

summary = pd.DataFrame(rows)
summary.to_csv(OUT_DIR / "nairobi_dl_only_summary.csv", index=False)
display(summary)
print("Saved:", OUT_DIR / "nairobi_dl_only_summary.csv")

# Keep gt_flood as combined union for backward-compatible exports.
gt_flood = gt_combined


In [ ]:
# Visualize DL-only outputs against both ground-truth references
from matplotlib.colors import BoundaryNorm, LinearSegmentedColormap, ListedColormap
from matplotlib.patches import Patch

FIG_DIR = OUT_DIR / "report_figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

TEAL = "#4db6ac"
TEAL_DARK = "#00695c"
ORANGE = "#ef6c02"
PROB_CMAP = LinearSegmentedColormap.from_list("white_to_teal", ["#f7fbff", TEAL_DARK])
CHANGE_CMAP = LinearSegmentedColormap.from_list("orange_white_teal", [ORANGE, "#f7f7f7", TEAL_DARK])
OVERLAY_CMAP = ListedColormap([ORANGE, TEAL, TEAL_DARK])
OVERLAY_NORM = BoundaryNorm([0.5, 1.5, 2.5, 3.5], OVERLAY_CMAP.N)

def polish_map(ax):
    ax.set_aspect("equal")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")

def save_dataarray_map(da, filename, title, cmap, vmin=None, vmax=None, add_colorbar=True):
    fig, ax = plt.subplots(figsize=(7.2, 6.2))
    da.plot(ax=ax, cmap=cmap, vmin=vmin, vmax=vmax, add_colorbar=add_colorbar)
    ax.set_title(title)
    polish_map(ax)
    plt.tight_layout()
    path = FIG_DIR / filename
    fig.savefig(path, dpi=220, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("Saved:", path)

def save_binary_mask(mask, filename, title, color):
    cmap = ListedColormap([color])
    fig, ax = plt.subplots(figsize=(7.2, 6.2))
    mask.where(mask == 1).plot(ax=ax, cmap=cmap, add_colorbar=False)
    ax.set_title(title)
    polish_map(ax)
    plt.tight_layout()
    path = FIG_DIR / filename
    fig.savefig(path, dpi=220, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("Saved:", path)

def make_overlay(dl_mask, gt_mask):
    dl = dl_mask.astype(bool)
    gt = gt_mask.astype(bool)
    overlay = xr.full_like(dl_mask.astype("int16"), fill_value=0)
    overlay = overlay.where(~(dl & ~gt), 1)
    overlay = overlay.where(~(~dl & gt), 2)
    overlay = overlay.where(~(dl & gt), 3)
    return overlay.where(overlay > 0)

def save_overlay(dl_mask, gt_mask, filename, title, gt_label):
    overlay = make_overlay(dl_mask, gt_mask)
    fig, ax = plt.subplots(figsize=(7.2, 6.2))
    overlay.plot(ax=ax, cmap=OVERLAY_CMAP, norm=OVERLAY_NORM, add_colorbar=False)
    ax.set_title(title)
    polish_map(ax)
    handles = [
        Patch(facecolor=ORANGE, edgecolor="none", label="DL flood proxy only"),
        Patch(facecolor=TEAL, edgecolor="none", label=f"{gt_label} only"),
        Patch(facecolor=TEAL_DARK, edgecolor="none", label="DL + GT overlap"),
    ]
    ax.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, -0.22), ncol=1, frameon=True)
    plt.tight_layout()
    path = FIG_DIR / filename
    fig.savefig(path, dpi=220, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("Saved:", path)

# Individual report figures
save_dataarray_map(before_water_prob, "01_dl_before_water_probability.png", "DL before-water probability", PROB_CMAP, 0, 1)
save_dataarray_map(after_water_prob, "02_dl_after_water_probability.png", "DL after-water probability", PROB_CMAP, 0, 1)
save_binary_mask(flood_dl_mask, "03_dl_flood_proxy.png", f"DL flood proxy >= {BEST_WATER_THRESHOLD:.2f}", ORANGE)
save_binary_mask(gt_groundsource, "04_groundsource_media_gt.png", "Groundsource/media flood reference", TEAL)
save_binary_mask(gt_unstats, "05_unstats_gt.png", "UNStats flood extent reference", TEAL)
save_binary_mask(gt_combined, "06_combined_gt_union.png", "Combined flood reference union", TEAL)
save_dataarray_map(prob_change, "07_dl_water_probability_change.png", "DL water probability change", CHANGE_CMAP, -1, 1)
save_overlay(flood_dl_mask, gt_groundsource, "08_overlay_groundsource.png", "DL flood proxy vs Groundsource/media reference", "Groundsource/media GT")
save_overlay(flood_dl_mask, gt_unstats, "09_overlay_unstats.png", "DL flood proxy vs UNStats reference", "UNStats GT")

# Compact quicklook for notebook review
fig, axes = plt.subplots(3, 3, figsize=(18, 15))
before_water_prob.plot(ax=axes[0, 0], cmap=PROB_CMAP, vmin=0, vmax=1)
axes[0, 0].set_title("DL before-water probability")
after_water_prob.plot(ax=axes[0, 1], cmap=PROB_CMAP, vmin=0, vmax=1)
axes[0, 1].set_title("DL after-water probability")
flood_dl_mask.where(flood_dl_mask == 1).plot(ax=axes[0, 2], cmap=ListedColormap([ORANGE]), add_colorbar=False)
axes[0, 2].set_title(f"DL flood proxy >= {BEST_WATER_THRESHOLD:.2f}")
gt_groundsource.where(gt_groundsource == 1).plot(ax=axes[1, 0], cmap=ListedColormap([TEAL]), add_colorbar=False)
axes[1, 0].set_title("Groundsource/media GT")
gt_unstats.where(gt_unstats == 1).plot(ax=axes[1, 1], cmap=ListedColormap([TEAL]), add_colorbar=False)
axes[1, 1].set_title("UNStats flood extent GT")
gt_combined.where(gt_combined == 1).plot(ax=axes[1, 2], cmap=ListedColormap([TEAL]), add_colorbar=False)
axes[1, 2].set_title("Combined GT union")
prob_change.plot(ax=axes[2, 0], cmap=CHANGE_CMAP, vmin=-1, vmax=1)
axes[2, 0].set_title("DL water probability change")
make_overlay(flood_dl_mask, gt_groundsource).plot(ax=axes[2, 1], cmap=OVERLAY_CMAP, norm=OVERLAY_NORM, add_colorbar=False)
axes[2, 1].set_title("Overlay vs Groundsource/media")
make_overlay(flood_dl_mask, gt_unstats).plot(ax=axes[2, 2], cmap=OVERLAY_CMAP, norm=OVERLAY_NORM, add_colorbar=False)
axes[2, 2].set_title("Overlay vs UNStats")
for ax in axes.ravel():
    polish_map(ax)
overlay_handles = [
    Patch(facecolor=ORANGE, edgecolor="none", label="DL only"),
    Patch(facecolor=TEAL, edgecolor="none", label="GT only"),
    Patch(facecolor=TEAL_DARK, edgecolor="none", label="Overlap"),
]
axes[2, 1].legend(handles=overlay_handles, loc="lower center", bbox_to_anchor=(0.5, -0.35), frameon=True)
axes[2, 2].legend(handles=overlay_handles, loc="lower center", bbox_to_anchor=(0.5, -0.35), frameon=True)
plt.tight_layout()
quicklook_path = OUT_DIR / "nairobi_dl_only_quicklook.png"
fig.savefig(quicklook_path, dpi=180, bbox_inches="tight")
plt.show()
plt.close(fig)
print("Saved:", quicklook_path)
print("Saved individual figures to:", FIG_DIR)


In [ ]:
# Optional GeoTIFF export
before_water_prob.rio.write_crs(f"EPSG:{EPSG}", inplace=True)
after_water_prob.rio.write_crs(f"EPSG:{EPSG}", inplace=True)
flood_dl_mask.astype("uint8").rio.write_crs(f"EPSG:{EPSG}", inplace=True)
gt_groundsource.astype("uint8").rio.write_crs(f"EPSG:{EPSG}", inplace=True)
gt_unstats.astype("uint8").rio.write_crs(f"EPSG:{EPSG}", inplace=True)
gt_combined.astype("uint8").rio.write_crs(f"EPSG:{EPSG}", inplace=True)

before_water_prob.rio.to_raster(OUT_DIR / "nairobi_before_water_probability.tif")
after_water_prob.rio.to_raster(OUT_DIR / "nairobi_after_water_probability.tif")
flood_dl_mask.astype("uint8").rio.to_raster(OUT_DIR / "nairobi_dl_flood_mask.tif")
gt_groundsource.astype("uint8").rio.to_raster(OUT_DIR / "nairobi_groundsource_flood_rasterized.tif")
gt_unstats.astype("uint8").rio.to_raster(OUT_DIR / "nairobi_unstats_flood_rasterized.tif")
gt_combined.astype("uint8").rio.to_raster(OUT_DIR / "nairobi_combined_groundtruth_flood_rasterized.tif")
print("Saved GeoTIFFs in", OUT_DIR)
